# 04 — Model Optimization and Evaluation

**Purpose.** Hyperparameter search for the LSTM, final comparison against baselines, confusion-matrix analysis, and error analysis.

**Selection metric:** validation macro-F1  
**Test set:** used once at the end for the best configuration


## 1. Setup


In [1]:
import sys
from pathlib import Path

REPO_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if str(REPO_ROOT / 'src') not in sys.path:
    sys.path.append(str(REPO_ROOT / 'src'))

import json
import itertools
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import ConfusionMatrixDisplay, classification_report

from campaign_strategist.config import ARTIFACT_DIR, CAMPAIGN_CLASSES, PROCESSED_DATA_DIR, PROJECT_ROOT
from campaign_strategist.baselines import flatten_sequences
from campaign_strategist.model import save_model
from campaign_strategist.training import classification_metrics, predict_labels, train_lstm, transform_sequences
from campaign_strategist.viz import BLUE, ORANGE, apply_report_style, clean_label, despine, save_figure

apply_report_style()
RANDOM_STATE = 7
np.random.seed(RANDOM_STATE)

## 2. Load Data and Splits from Notebook 03


In [2]:
x = np.load(PROCESSED_DATA_DIR / 'sequences_x.npy')
y = np.load(PROCESSED_DATA_DIR / 'labels_y.npy')
sample_index = pd.read_parquet(PROCESSED_DATA_DIR / 'sample_index.parquet')
scaler = joblib.load(ARTIFACT_DIR / 'sequence_scaler.joblib')

train_idx = np.load(ARTIFACT_DIR / 'train_idx.npy')
val_idx = np.load(ARTIFACT_DIR / 'val_idx.npy')
test_idx = np.load(ARTIFACT_DIR / 'test_idx.npy')

x_train = transform_sequences(x[train_idx], scaler)
x_val = transform_sequences(x[val_idx], scaler)
x_test = transform_sequences(x[test_idx], scaler)
y_train, y_val, y_test = y[train_idx], y[val_idx], y[test_idx]

print(f'train {len(train_idx):,} | val {len(val_idx):,} | test {len(test_idx):,}')

train 18,392 | val 4,437 | test 5,670


## 3. Hyperparameter Grid Search

We search a small grid over hidden size and learning rate. Sequence length stays fixed at 12 for this notebook because changing it would require rebuilding samples; that experiment can be added later if needed. Each run uses early stopping on validation loss; the ranking metric is **validation macro-F1**.


In [3]:
grid = {
    'hidden_size': [64, 96, 128],
    'learning_rate': [5e-4, 8e-4, 1e-3],
}

rows = []
for hidden_size, learning_rate in itertools.product(grid['hidden_size'], grid['learning_rate']):
    model, history = train_lstm(
        x_train, y_train, x_val, y_val,
        hidden_size=hidden_size,
        learning_rate=learning_rate,
        epochs=20,
        batch_size=64,
        patience=4,
        random_state=RANDOM_STATE,
    )
    val_pred = predict_labels(model, x_val)
    metrics = classification_metrics(y_val, val_pred)
    rows.append({
        'hidden_size': hidden_size,
        'learning_rate': learning_rate,
        'epochs_run': int(history[-1]['epoch']),
        'best_val_loss': float(min(h['val_loss'] for h in history)),
        'val_accuracy': metrics['accuracy'],
        'val_macro_f1': metrics['macro_f1'],
    })
    print(rows[-1])

grid_results = pd.DataFrame(rows).sort_values('val_macro_f1', ascending=False).reset_index(drop=True)
grid_results

{'hidden_size': 64, 'learning_rate': 0.0005, 'epochs_run': 11, 'best_val_loss': 1.3015094995498657, 'val_accuracy': 0.4106378183457291, 'val_macro_f1': 0.4769284659664592}
{'hidden_size': 64, 'learning_rate': 0.0008, 'epochs_run': 11, 'best_val_loss': 1.3001954555511475, 'val_accuracy': 0.418526031102096, 'val_macro_f1': 0.48570983889547614}
{'hidden_size': 64, 'learning_rate': 0.001, 'epochs_run': 19, 'best_val_loss': 1.2995864152908325, 'val_accuracy': 0.415596123506874, 'val_macro_f1': 0.4845033374179899}
{'hidden_size': 96, 'learning_rate': 0.0005, 'epochs_run': 15, 'best_val_loss': 1.3074597120285034, 'val_accuracy': 0.421455938697318, 'val_macro_f1': 0.4853754794900968}
{'hidden_size': 96, 'learning_rate': 0.0008, 'epochs_run': 19, 'best_val_loss': 1.3129998445510864, 'val_accuracy': 0.4135677259409511, 'val_macro_f1': 0.4787810350783568}
{'hidden_size': 96, 'learning_rate': 0.001, 'epochs_run': 12, 'best_val_loss': 1.3071353435516357, 'val_accuracy': 0.4054541356772594, 'val_mac

,hidden_size,learning_rate,epochs_run,best_val_loss,val_accuracy,val_macro_f1
0,64,0.0008,11,1.300195,0.418526,0.485710
1,96,0.0005,15,1.307460,0.421456,0.485375
2,128,0.0005,12,1.301194,0.413117,0.484559
3,64,0.0010,19,1.299586,0.415596,0.484503
4,128,0.0008,11,1.306888,0.414018,0.479207
5,96,0.0008,19,1.313000,0.413568,0.478781
6,64,0.0005,11,1.301509,0.410638,0.476928
7,128,0.0010,12,1.301748,0.405454,0.476169
8,96,0.0010,12,1.307135,0.405454,0.474771


## 4. Search Results


In [4]:
fig, ax = plt.subplots(figsize=(8, 4.5))
labels = [f"h={r.hidden_size}, lr={r.learning_rate:g}" for r in grid_results.itertuples()]
ax.barh(labels[::-1], grid_results['val_macro_f1'][::-1], color=BLUE)
ax.set_xlabel('Validation macro-F1')
ax.set_title('LSTM Hyperparameter Search')
despine(ax)
plt.tight_layout()
save_figure(fig, PROJECT_ROOT / 'reports' / 'figures' / 'optimization_macro_f1.png')

grid_results.to_csv(PROJECT_ROOT / 'reports' / 'tables' / 'optimization_results.csv', index=False)
best = grid_results.iloc[0].to_dict()
(ARTIFACT_DIR / 'best_hyperparameters.json').write_text(json.dumps(best, indent=2))
print('Best config:', best)
grid_results

Best config: {'hidden_size': 64.0, 'learning_rate': 0.0008, 'epochs_run': 11.0, 'best_val_loss': 1.3001954555511475, 'val_accuracy': 0.418526031102096, 'val_macro_f1': 0.48570983889547614}


,hidden_size,learning_rate,epochs_run,best_val_loss,val_accuracy,val_macro_f1
0,64,0.0008,11,1.300195,0.418526,0.485710
1,96,0.0005,15,1.307460,0.421456,0.485375
2,128,0.0005,12,1.301194,0.413117,0.484559
3,64,0.0010,19,1.299586,0.415596,0.484503
4,128,0.0008,11,1.306888,0.414018,0.479207
5,96,0.0008,19,1.313000,0.413568,0.478781
6,64,0.0005,11,1.301509,0.410638,0.476928
7,128,0.0010,12,1.301748,0.405454,0.476169
8,96,0.0010,12,1.307135,0.405454,0.474771


## 5. Final Model: Retrain Best Configuration

Retrain on train+validation using the best hyperparameters, then evaluate **once** on the held-out test set.


In [5]:
x_trainval = np.concatenate([x_train, x_val], axis=0)
y_trainval = np.concatenate([y_train, y_val], axis=0)

# Use a small slice of train as internal early-stopping monitor to avoid touching test.
rng = np.random.default_rng(RANDOM_STATE)
perm = rng.permutation(len(x_trainval))
cut = max(int(0.1 * len(perm)), 1)
inner_val = perm[:cut]
inner_train = perm[cut:]

final_model, final_history = train_lstm(
    x_trainval[inner_train], y_trainval[inner_train],
    x_trainval[inner_val], y_trainval[inner_val],
    hidden_size=int(best['hidden_size']),
    learning_rate=float(best['learning_rate']),
    epochs=25,
    batch_size=64,
    patience=5,
    random_state=RANDOM_STATE,
)

y_pred = predict_labels(final_model, x_test)
final_metrics = classification_metrics(y_test, y_pred)
print('Final test metrics:', {k: final_metrics[k] for k in ('accuracy', 'macro_f1')})

Final test metrics: {'accuracy': 0.44144620811287477, 'macro_f1': 0.5013681265551104}


## 6. Confusion Matrix and Per-Class Metrics


In [6]:
display_labels = [clean_label(c) for c in CAMPAIGN_CLASSES]
fig, ax = plt.subplots(figsize=(8, 6))
ConfusionMatrixDisplay.from_predictions(
    y_test, y_pred, display_labels=display_labels, xticks_rotation=45, cmap='Blues', ax=ax, colorbar=False
)
ax.set_title('Tuned LSTM Confusion Matrix (Test Set)')
plt.tight_layout()
save_figure(fig, PROJECT_ROOT / 'reports' / 'figures' / 'model_confusion_matrix.png')
print(classification_report(y_test, y_pred, target_names=CAMPAIGN_CLASSES, zero_division=0))

                         precision    recall  f1-score   support

new_customer_onboarding       0.96      1.00      0.98       329
      win_back_reminder       0.41      0.56      0.47      1035
       price_led_coupon       0.43      0.26      0.32      1660
      cross_sell_bundle       0.44      0.40      0.42      1565
         loyalty_reward       0.27      0.27      0.27       569
     seasonal_spotlight       0.42      0.75      0.54       512

               accuracy                           0.44      5670
              macro avg       0.49      0.54      0.50      5670
           weighted avg       0.44      0.44      0.43      5670



## 7. Error Analysis

We inspect the most confused class pairs and sample misclassified households. Errors between behaviorally similar styles (for example price-led coupon vs. loyalty reward) are more acceptable than errors between opposite strategies such as onboarding vs. win-back.


In [7]:
cm = np.asarray(final_metrics['confusion_matrix'])
pairs = []
for i in range(len(CAMPAIGN_CLASSES)):
    for j in range(len(CAMPAIGN_CLASSES)):
        if i != j and cm[i, j] > 0:
            pairs.append((cm[i, j], CAMPAIGN_CLASSES[i], CAMPAIGN_CLASSES[j]))
pairs = sorted(pairs, reverse=True)[:8]
print('Top confused pairs (true -> predicted):')
for count, true_c, pred_c in pairs:
    print(f'  {count:4d}  {true_c} -> {pred_c}')

test_index = sample_index.iloc[test_idx].reset_index(drop=True).copy()
test_index['y_true'] = y_test
test_index['y_pred'] = y_pred
errors = test_index[test_index.y_true != test_index.y_pred].head(10)
errors.assign(
    true_label=lambda d: d.y_true.map(lambda i: CAMPAIGN_CLASSES[i]),
    pred_label=lambda d: d.y_pred.map(lambda i: CAMPAIGN_CLASSES[i]),
)[['household_id', 'end_week', 'true_label', 'pred_label']]

Top confused pairs (true -> predicted):
   429  price_led_coupon -> cross_sell_bundle
   356  price_led_coupon -> win_back_reminder
   332  cross_sell_bundle -> win_back_reminder
   294  cross_sell_bundle -> price_led_coupon
   244  price_led_coupon -> seasonal_spotlight
   197  price_led_coupon -> loyalty_reward
   191  loyalty_reward -> cross_sell_bundle
   161  cross_sell_bundle -> loyalty_reward


,household_id,end_week,true_label,pred_label
0,1000,27,cross_sell_bundle,loyalty_reward
5,1000,37,cross_sell_bundle,loyalty_reward
6,1000,38,cross_sell_bundle,loyalty_reward
12,1000,49,cross_sell_bundle,seasonal_spotlight
17,1002,46,seasonal_spotlight,price_led_coupon
18,1002,47,seasonal_spotlight,price_led_coupon
19,1002,48,seasonal_spotlight,price_led_coupon
20,1002,49,seasonal_spotlight,price_led_coupon
21,1005,12,price_led_coupon,loyalty_reward
23,1005,14,price_led_coupon,loyalty_reward


## 8. Methodology Verification: Shuffled-Label Control

To confirm that the observed performance comes from real signal rather than a bug or hidden leakage, the same LSTM is trained on **randomly shuffled training labels**. If the pipeline leaked information anywhere (labels visible in features, households crossing splits, scaler fit on test data), the model could still score above chance on this control. A collapse to chance-level macro-F1 (~0.17 for six classes) confirms the methodology is sound: the model can only perform when the true label-behavior relationship exists.

In [8]:
rng = np.random.default_rng(RANDOM_STATE)
y_train_shuffled = rng.permutation(y_train)

control_model, _ = train_lstm(
    x_train, y_train_shuffled, x_val, y_val,
    hidden_size=int(best['hidden_size']),
    learning_rate=float(best['learning_rate']),
    epochs=8,
    batch_size=64,
    patience=8,  # no early stop: let it try its best on noise
    random_state=RANDOM_STATE,
)

control_pred = predict_labels(control_model, x_test)
control_metrics = classification_metrics(y_test, control_pred)

print(f"Shuffled-label control : accuracy {control_metrics['accuracy']:.3f} | "
      f"macro-F1 {control_metrics['macro_f1']:.3f}")
print(f"Real-label final model : accuracy {final_metrics['accuracy']:.3f} | "
      f"macro-F1 {final_metrics['macro_f1']:.3f}")
print(f"Chance level (6 classes): macro-F1 ~0.17 with balanced random guessing")

Shuffled-label control : accuracy 0.265 | macro-F1 0.107
Real-label final model : accuracy 0.441 | macro-F1 0.501
Chance level (6 classes): macro-F1 ~0.17 with balanced random guessing


## 9. Performance Analysis: What Limits the Score, and What Would Improve It

The final macro-F1 of ~0.50 should be read against three structural constraints, none of which is a bug:

1. **Label noise is the ceiling.** The labels are weak-supervision rules, approximate by design. A model cannot exceed the consistency of its labels; part of every 'error' is disagreement with an imperfect rule, not a wrong marketing decision. The abstention rate (53% of windows produce no clear label) shows how often real behavior is ambiguous.
2. **Neighboring classes genuinely overlap.** The largest confusions (price-led ↔ cross-sell, cross-sell ↔ win-back) occur between behaviorally similar activation styles. For a marketer, recommending a cross-sell bundle to a price-sensitive household is a much smaller mistake than onboarding vs. win-back — and the model makes almost no mistakes of the second kind.
3. **The data is sparse.** The public transaction sample gives a median of 13 active weeks per household per year, so many 12-week windows contain only 2-4 shopping events.

Concrete steps that would plausibly improve performance, in order of expected impact:

- **Real campaign-response labels** from the Complete Journey `campaigns`/`coupon_redemptions` tables, replacing weak supervision for households that received real campaigns.
- **Full transaction data** instead of the published sample (~10x more transactions per household would densify the weekly sequences).
- **Household demographics** (income band, household size) as static features next to the behavioral sequences.
- **Longer windows or hierarchical models** once denser data makes them estimable.

## 10. Final Comparison Table for the Report


In [9]:
# Recompute baselines on the same held-out test set for a fair final table.
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier

f_train = flatten_sequences(x_train)
f_test = flatten_sequences(x_test)
knn = KNeighborsClassifier(n_neighbors=15, weights='distance').fit(f_train, y_train)
rf = RandomForestClassifier(
    n_estimators=300, max_depth=12, min_samples_leaf=3,
    class_weight='balanced', random_state=RANDOM_STATE, n_jobs=1,
).fit(f_train, y_train)

rows = [
    {'model': 'LSTM (tuned)', **{k: final_metrics[k] for k in ('accuracy', 'macro_f1')}},
    {'model': 'random_forest', **{k: classification_metrics(y_test, rf.predict(f_test))[k] for k in ('accuracy', 'macro_f1')}},
    {'model': 'knn', **{k: classification_metrics(y_test, knn.predict(f_test))[k] for k in ('accuracy', 'macro_f1')}},
]
final_table = pd.DataFrame(rows).sort_values('macro_f1', ascending=False).reset_index(drop=True)
final_table.to_csv(PROJECT_ROOT / 'reports' / 'tables' / 'model_comparison.csv', index=False)

fig, ax = plt.subplots(figsize=(7, 4.5))
final_table.set_index('model')[['accuracy', 'macro_f1']].plot(kind='bar', ax=ax, color=[BLUE, ORANGE], rot=0)
ax.set_ylim(0, 1)
ax.set_ylabel('Score')
ax.set_title('Final Model Comparison (Test Set)')
despine(ax)
plt.tight_layout()
save_figure(fig, PROJECT_ROOT / 'reports' / 'figures' / 'model_comparison.png')

save_model(final_model, str(ARTIFACT_DIR / 'journey_lstm.pt'))
(ARTIFACT_DIR / 'nb04_metrics.json').write_text(json.dumps({
    'best_hyperparameters': best,
    'final_test': {k: final_metrics[k] for k in ('accuracy', 'macro_f1')},
    'comparison': final_table.to_dict(orient='records'),
}, indent=2))
final_table

,model,accuracy,macro_f1
0,LSTM (tuned),0.441446,0.501368
1,random_forest,0.404586,0.436955
2,knn,0.367019,0.399616


## 11. Conclusions

The tuned LSTM is evaluated against KNN and Random Forest on a household-held-out test set. Because labels are weak-supervision signals from future purchase behavior rather than true campaign response outcomes, strong metrics support the modeling approach but should not be over-interpreted as causal campaign lift. Limitations include the public sample size, retailer specificity, and rule-based labels. Future work could replace weak labels with approved campaign-response outcomes and test additional seasonal windows.

## References

Hochreiter, S., & Schmidhuber, J. (1997). Long short-term memory. *Neural Computation, 9*(8), 1735–1780.

Paszke, A., et al. (2019). PyTorch: An imperative style, high-performance deep learning library. *Advances in Neural Information Processing Systems, 32*.

Pedregosa, F., et al. (2011). Scikit-learn: Machine learning in Python. *Journal of Machine Learning Research, 12*, 2825–2830.

Ratner, A., Bach, S. H., Ehrenberg, H., Fries, J., Wu, S., & Ré, C. (2017). Snorkel: Rapid training data creation with weak supervision. *Proceedings of the VLDB Endowment, 11*(3), 269–282.
